# Inspect stage-1 artifacts

Quick sanity check + model-readiness smoke test for the prepared FlyWire FAFB v783 connectome.
Run with the `consortium` kernel (or any env with the `flyconn` package installed).

In [ ]:
from flyconn.config import load_config
from flyconn.io import read_parquet, load_csr

cfg = load_config()
p = cfg.paths()
neurons = read_parquet(p.neurons)
edges = read_parquet(p.edges)
print('neurons:', len(neurons), 'edges:', len(edges))
neurons.head()

In [ ]:
# signed adjacency (default policy): A[i, j] = weight of edge i -> j
A = load_csr(p.adjacency_signed(cfg.default_nt_policy))
print('shape', A.shape, 'nnz', A.nnz)
print('positive mass', A.data[A.data > 0].sum(), 'negative mass', A.data[A.data < 0].sum())
assert A.shape[0] == len(neurons)  # rows align with node table

In [ ]:
# model-ready torch sparse tensor
import torch
blob = torch.load(p.adjacency_pt)
print({k: v for k, v in blob.items() if k != 'adj'})
adj = blob['adj']  # torch sparse CSR, shape (N, N)
print(adj.shape, adj._nnz() if hasattr(adj, '_nnz') else 'n/a')
# For an RNN recurrent weight matrix W with r_next = W @ r, transpose: W = adj.t()

In [ ]:
# out-degree of a neuron, spot-check against Codex (codex.flywire.ai)
import numpy as np
out_deg = np.asarray((load_csr(p.adjacency_counts) != 0).sum(axis=1)).ravel()
top = neurons.assign(out_partners=out_deg).nlargest(5, 'out_partners')
top[['idx', 'root_id', 'cell_type', 'super_class', 'out_partners']]